# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

  Using cached deepspeed-0.18.9.tar.gz (1.7 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  python setup.py egg_info did not run successfully.
  exit code: 1
  
  [33 lines of output]
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
    File "<pip-setuptools-caller>", line 34, in <module>
    File "C:\Users\CHERT\AppData\Local\Temp\pip-install-55ngowc4\deepspeed_f26ac784d53844f6a926c05ecc479ed5\setup.py", line 40, in <module>
      from op_builder import get_default_compute_capabilities, OpBuilder
    File "C:\Users\CHERT\AppData\Local\Temp\pip-install-55ngowc4\deepspeed_f26ac784d53844f6a926c05ecc479ed5\op_builder\__init__.py", line 18, in <module>
      import deepspeed.ops.op_builder  # noqa: F401 # type: ignore
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "C:\Users\CHERT\AppData\Local\Temp\pip-install-55ngowc4\deepspeed_f26ac784d53844f6a926c05ecc479ed5\deepspeed\__init__.py", line 25, in <module>
      from . import ops
    File "C:\Users\CHERT\AppData\Local\Temp\pip-install-55ngowc4\deepspeed_f26ac7

### Data Preparation

In [2]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [5]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [7]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [8]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [9]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        inputs = {
            'input_ids': batch['input_ids'].to(device),
            'attention_mask': batch['attention_mask'].to(device),
            'token_type_ids': batch['token_type_ids'].to(device)
        }
        
        labels = batch['labels'].to(device)
        
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        correct += (predictions == labels).sum().item()
        total += predictions.size(0)

accuracy = correct / total
print(f"Validation Accuracy: {accuracy:.4f}")

Evaluating: 100%|███████████████████████████████████████████████████████████████| 40430/40430 [04:37<00:00, 145.95it/s]

Validation Accuracy: 0.9084


In [10]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import numpy as np
import evaluate

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

def preprocess_function(examples):
    return tokenizer(examples["text1"], examples["text2"], truncation=True, max_length=128)

encoded_dataset = qqp.map(preprocess_function, batched=True)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results_distilbert",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none"
)

train_subset = encoded_dataset["train"].shuffle(seed=42).select(range(15000))
eval_subset = encoded_dataset["validation"].shuffle(seed=42).select(range(2000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training DistilBERT...")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda
Starting training DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.397322,0.810500
2,0.465286,0.405107,0.827500
3,0.311870,0.441310,0.824500


TrainOutput(global_step=1407, training_loss=0.3429638741282427, metrics={'train_runtime': 75.7317, 'train_samples_per_second': 594.203, 'train_steps_per_second': 18.579, 'total_flos': 795145909448160.0, 'train_loss': 0.3429638741282427, 'epoch': 3.0})

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [12]:
def find_duplicates(question, dataset, model, tokenizer, top_k=5, search_limit=1000):

    device = model.device
    model.eval()
 
    candidates = dataset.select(range(min(len(dataset), search_limit)))
    candidate_texts = candidates["text1"] 

    pairs = [[question, cand] for cand in candidate_texts]
    
    duplicates = []
    batch_size = 32
    
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i : i + batch_size]
            inputs = tokenizer(
                [p[0] for p in batch_pairs], 
                [p[1] for p in batch_pairs], 
                padding=True, truncation=True, max_length=128, return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)

            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            
            for j, prob in enumerate(probs):
                if prob.item() > 0.1:
                    duplicates.append((candidate_texts[i + j], prob.item()))
    
    duplicates.sort(key=lambda x: x[1], reverse=True)
    return duplicates[:top_k]


test_questions = [
    "How can I learn Python?",
    "What is the best way to lose weight?",
    "Why is the sky blue?",
    "How do I make money online?",
    "What are the best movies of 2020?"
]

search_dataset = qqp["train"]

print(f"Searching duplicates in first {1000} examples...\n")

for q in test_questions:
    print(f"Query: {q}")
    results = find_duplicates(q, search_dataset, model, tokenizer)
    
    if not results:
        print("  No duplicates found in subset.")
    for res, score in results:
        print(f"  [{score:.4f}] {res}")
    print("-" * 30)

Searching duplicates in first 1000 examples...

Query: How can I learn Python?
  No duplicates found in subset.
------------------------------
Query: What is the best way to lose weight?
  [0.9372] How do I lose weight fast?
  [0.9371] How do I lose weight fast?
  [0.9266] Which foods help gain weight?
  [0.9078] How can I lose 4kg weight?
  [0.8896] How do I lose 25 kg by exercise?
------------------------------
Query: Why is the sky blue?
  No duplicates found in subset.
------------------------------
Query: How do I make money online?
  [0.9651] What are your views about governments decision to stop flow of 1000 and 500 rupee notes.?
  [0.9628] How can changing 500 and 1000 rupee notes end the black money in India?
  [0.9573] How do I really make money online?
  [0.9549] What is the easy way to make money online?
  [0.9472] What will be the repercussions of banning Rs 500 and Rs 1000 notes on Indian economy?
------------------------------
Query: What are the best movies of 2020?
  [

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>